# Full `ADF conversion` Prefect Flow

This demo presents the implementation for the following stories:

  * https://pforge-exchange2.astrium.eads.net/jira/browse/RSPY-1014
  * https://pforge-exchange2.astrium.eads.net/jira/browse/RSPY-1015
  * https://pforge-exchange2.astrium.eads.net/jira/browse/RSPY-1016
  * https://pforge-exchange2.astrium.eads.net/jira/browse/RSPY-1019
  * https://pforge-exchange2.astrium.eads.net/jira/browse/RSPY-1023     

## Initialisation

In [ ]:
# Imports
import ast
from IPython.display import JSON
import os
import os.path as osp

from resources.widget_utils import *

from rs_common.prefect_utils import *
from rs_workflows.aux_flow import aux_staging
from rs_workflows.flow_utils import  DprProcessIn, Priority, ProcessingMode, WorkflowType
from rs_workflows.init_pi_db_flow import init_pi_database
from rs_workflows.adf_flow import adf_conversion

In [ ]:
print(f"Prefect server URL used internally: {os.environ['PREFECT_API_URL']}")
dashboard_url = f"{os.environ['RSPY_PREFECT_URL']}/dashboard"
print(f"Prefect dashboard public URL: {dashboard_url}")

In [ ]:
# Choose prefect deployment method
deploy_prefect_radio

In [ ]:
# Choose prefect flow run method
run_prefect_radio

In [ ]:
# Init environment before running a demo notebook.
from resources.utils import *
init_demo()
# Reload the global vars again
from resources.utils import *

from resources.dask_clusters.dask_main_env import *
await init_dask_cluster_staging()

In [ ]:
# Create test collections
from rs_workflows.flow_utils import AdfProcessIn
AUXIP_COLLECTION = "TEST_FLOW_AUXIP"
STAGE_COLLECTION = "TEST_STAGE_ADF_CONVERSION"
        
create_test_collection(AUXIP_COLLECTION)
create_test_collection(STAGE_COLLECTION)
  
# Prefect flow environment arguments
flow_env_args = {
  "env": {
    "owner_id": OWNER_ID,
  },
}



## Deploy rs-client-libraries Prefect flows

In [ ]:
# Deploy the Prefect flows
s3_code_folder = f"users/{OWNER_ID}/code"
adf_conversion_deploy = await deploy_prefect(
    deploy_file="./adf_conversion_flow.yaml", 
    s3_code_folder=s3_code_folder, 
    work_pool_name=os.environ["PREFECT_WORK_POOL_SANDBOX"]
)

## Run the ADF conversion flow

In [ ]:
# Choose adf type to convert
adf_proc_radio

In [ ]:
print(f"Runing the demo for: {adf_proc_radio.value!r}")
from rs_workflows.flow_utils import AdfType
from rs_workflows.adf_flow import S03_OL_ADF_TYPE_CONFIG

# one collection may be used instead of these two, but for the demo we want to show the mapping 
# functionality, where different product types can be mapped to different collections
# if one collection is used, the collection name should be the same for all product types in the mapping like this: 
# auxiliary_product_to_collection_identifier = [{"product_type": "*", "collection_name": AUXIP_COLLECTION}]
match adf_proc_radio.value:
    case  AdfType.S00__ADF_ECMWA:
        auxiliary_product_to_collection_identifier = [{"product_type": "AX___MA1_AX", "collection_name": STAGE_COLLECTION},
                                                 {"product_type": "ADF_ECMWA", "collection_name": AUXIP_COLLECTION}]
    case AdfType.S00__ADF_ECMWF:
        auxiliary_product_to_collection_identifier = [{"product_type": "AX___MF1_AX", "collection_name": STAGE_COLLECTION},
                                                 {"product_type": "ADF_ECMWF", "collection_name": AUXIP_COLLECTION}]
    case AdfType.S00__ADF_GETAS:
        auxiliary_product_to_collection_identifier = [{"product_type": "AX___DEM_AX", "collection_name": STAGE_COLLECTION},
                                                 {"product_type": "ADF_GETAS", "collection_name": AUXIP_COLLECTION}]
    case AdfType.S00__ADF_WATER:
        auxiliary_product_to_collection_identifier = [{"product_type": "*", "collection_name": STAGE_COLLECTION},
                                                 {"product_type": "ADF_WATER", "collection_name": AUXIP_COLLECTION}]
    case adf_type if adf_type in S03_OL_ADF_TYPE_CONFIG:        
        auxiliary_product_to_collection_identifier = [{"product_type": "*", "collection_name": STAGE_COLLECTION},
                                                 {"product_type": S03_OL_ADF_TYPE_CONFIG[adf_type].generated_prod_type, "collection_name": AUXIP_COLLECTION}]
    case _:
        print(f"Unsupported adf_type: {adf_proc_radio.value}")
        sys.exit(1)        
# DPR processing input parameters
adg_process_in = AdfProcessIn(
    **flow_env_args,         
    adf_type=adf_proc_radio.value,    
    auxiliary_product_to_collection_identifier = auxiliary_product_to_collection_identifier,
    start_datetime="2026-01-01T11:00:00Z",
    end_datetime="2026-04-03T11:00:00Z",
    satellite=None,
)
# Run the processor
params = {"adf_input": adg_process_in.model_dump(mode="json")}
state = await run_prefect(
    deploy_name=adf_conversion_deploy, 
    py_func=adf_conversion, 
    params=params
)
if state is not None:
    flow_run_id = state.state_details.flow_run_id
    print(f"Flow run id: {flow_run_id!r}")
    if state.is_failed() or state.is_crashed():
        # Retrieve logs of failed flow run
        response = http_session.get(f"{os.environ['PREFECT_API_URL']}/flow_runs/{flow_run_id}/logs/download")
        response.raise_for_status()
        logs_text = response.text

        print("=== Prefect flow logs ===")
        print(logs_text)
        print("=== End of logs ===")

        raise RuntimeError(f"Prefect flow {flow_run_id} failed.\n\nLogs:\n{logs_text}")